# Quantized Retrieval-Augmented Medical VLM Diagnosis - Kaggle Full Test

Notebook này chạy full workflow trên Kaggle và lưu artifacts để dùng lại:

1. Clone hoặc pull repo mới nhất từ GitHub.
2. Kiểm tra GPU/CUDA.
3. Cài dependency nhưng không cài đè PyTorch của Kaggle.
4. Kiểm tra dataset IU Chest X-rays.
5. Chạy demo nhanh nếu cần.
6. Train/evaluate 10 epoch với log ra file.
7. Inspect metrics/model artifacts.
8. Đóng gói artifacts thành `.zip` để tải về hoặc attach làm Kaggle Dataset cho lần sau.


## 0. Central Test Configuration

Chỉnh các biến dưới đây trước khi chạy. Mặc định notebook dùng GitHub repo của dự án và dataset IU Chest X-rays path bạn đang dùng.


In [ ]:
from pathlib import Path

# Source options: "auto", "current", "github", or "kaggle_dataset"
SOURCE_MODE = "github"
GITHUB_REPO_URL = "https://github.com/ChienNguyensrdn/medical_vlm_pipeline.git"
KAGGLE_SOURCE_DIR = "/kaggle/input/medical-vlm-pipeline"
FORCE_FRESH_CLONE = False  # True = delete /kaggle/working repo and clone from scratch
RESET_REPO_BEFORE_PULL = True  # Kaggle-safe: discard generated files before pulling latest source
GITHUB_BRANCH = "main"

# Kaggle GPU/PyTorch config
# Enable this when Kaggle preinstalled torch reports CUDA but fails on Tesla T4 with
# "CUDA error: no kernel image is available for execution on the device".
REINSTALL_TORCH_FOR_T4 = True
PYTORCH_CUDA_INDEX_URL = "https://download.pytorch.org/whl/cu121"
PYTORCH_PACKAGES = ["torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1"]

# Dataset config
DATASET_DIR = "/kaggle/input/datasets/masrursabab/iu-chest-x-rays-cleaned"
IMAGE_FOLDER = "resized_images/256"

# Full-test training config
EPOCHS = 150
BATCH_SIZE = 16  # global batch; DataParallel splits this across T4x2 as 8+8
SYNTHETIC_CASES = 32
DEVICE = "cuda"  # cuda is intentional for Kaggle T4; use auto/cpu only for debugging
TARGET_COLUMN = "auto"
DERIVE_LABELS_FROM_REPORT = True  # weak pathology labels instead of Frontal/Lateral projection
MIN_CLASS_COUNT = 16
CLASS_WEIGHTING = "balanced"  # balanced or none; helps weak-label class imbalance
VAL_SPLIT = 0.15
SPLIT_GROUP = "report"  # report prevents exact-report leak; on IU it also keeps paired views together
NUM_WORKERS = 2
MULTI_GPU = "auto"  # auto/data_parallel/off; auto uses T4x2 when visible
GPU_IDS = "0,1"  # Kaggle T4x2 device ids
NO_AMP = False

# Toggle stages
INSTALL_DEPS = True
RUN_DEMO = True
RUN_FULL_TRAIN = True
SKIP_PLOT = False

PROJECT_ROOT = Path("/kaggle/working/medical_vlm_pipeline")
SRC_DIR = PROJECT_ROOT / "src"
print("Config ready")


## 1. Clone Or Pull Source Code

Nếu `/kaggle/working/medical_vlm_pipeline` chưa có, cell này sẽ clone repo. Nếu đã có, cell này sẽ `git fetch` + `git pull --ff-only` để lấy code mới nhất.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path


def is_src_dir(path: Path) -> bool:
    return (path / "train_pipeline.py").exists() and (path / "medical_vlm_pipeline").exists()


def normalize_to_src(path: Path) -> Path | None:
    path = Path(path)
    if is_src_dir(path):
        return path.resolve()
    if is_src_dir(path / "src"):
        return (path / "src").resolve()
    return None


def copy_source_to_working(source_path: Path) -> Path:
    source_src = normalize_to_src(source_path)
    if source_src is None:
        raise FileNotFoundError(f"No train_pipeline.py found under source path: {source_path}")
    source_root = source_src.parent if source_src.name == "src" else source_src
    target_root = PROJECT_ROOT
    if target_root.exists():
        shutil.rmtree(target_root)
    shutil.copytree(source_root, target_root)
    copied_src = normalize_to_src(target_root)
    if copied_src is None:
        raise FileNotFoundError(f"Copied source, but no runnable src folder found at: {target_root}")
    return copied_src


def run_git(target_root: Path, *args: str) -> None:
    cmd = ["git", "-C", str(target_root), *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


def clone_or_pull_repo(repo_url: str, target_root: Path) -> Path:
    os.chdir("/kaggle/working")
    branch = GITHUB_BRANCH
    if target_root.exists() and FORCE_FRESH_CLONE:
        print(f"FORCE_FRESH_CLONE=True, removing: {target_root}")
        shutil.rmtree(target_root)
    if not target_root.exists():
        cmd = ["git", "clone", "--branch", branch, repo_url, str(target_root)]
        print("Cloning:", " ".join(cmd))
        subprocess.run(cmd, check=True)
    else:
        if not (target_root / ".git").exists():
            print(f"Existing folder is not a git repo, replacing: {target_root}")
            shutil.rmtree(target_root)
            subprocess.run(["git", "clone", "--branch", branch, repo_url, str(target_root)], check=True)
        else:
            print(f"Existing repo found: {target_root}")
            run_git(target_root, "remote", "set-url", "origin", repo_url)
            run_git(target_root, "fetch", "--prune", "origin", branch)
            print("Local status before source refresh:")
            subprocess.run(["git", "-C", str(target_root), "status", "--short"], check=True)
            if RESET_REPO_BEFORE_PULL:
                run_git(target_root, "reset", "--hard", f"origin/{branch}")
                run_git(target_root, "clean", "-fd", "--", "src", ".")
            else:
                run_git(target_root, "checkout", branch)
                run_git(target_root, "pull", "--ff-only", "origin", branch)
    print("Current source commit after refresh:")
    subprocess.run(["git", "-C", str(target_root), "log", "-1", "--oneline"], check=True)
    print("Current remote HEAD:")
    subprocess.run(["git", "-C", str(target_root), "rev-parse", f"origin/{branch}"], check=True)
    print("Current local HEAD:")
    subprocess.run(["git", "-C", str(target_root), "rev-parse", "HEAD"], check=True)
    src = normalize_to_src(target_root)
    if src is None:
        raise FileNotFoundError(f"Repo is present but src folder was not found under: {target_root}")
    return src


if SOURCE_MODE == "github":
    SRC_DIR = clone_or_pull_repo(GITHUB_REPO_URL, PROJECT_ROOT)
elif SOURCE_MODE == "kaggle_dataset":
    SRC_DIR = copy_source_to_working(Path(KAGGLE_SOURCE_DIR))
else:
    candidates = [
        Path.cwd(), Path.cwd() / "src",
        Path("/kaggle/working"), Path("/kaggle/working/src"),
        Path("/kaggle/working/medical_vlm_pipeline"), Path("/kaggle/working/medical_vlm_pipeline/src"),
        Path(KAGGLE_SOURCE_DIR), Path(KAGGLE_SOURCE_DIR) / "src",
    ]
    selected = None
    for candidate in candidates:
        normalized = normalize_to_src(candidate)
        if normalized is not None:
            selected = normalized
            break
    if selected is None and SOURCE_MODE == "auto":
        for match in Path("/kaggle/input").glob("**/train_pipeline.py"):
            normalized = normalize_to_src(match.parent)
            if normalized is not None:
                selected = normalized
                break
    if selected is None:
        raise FileNotFoundError("Cannot find train_pipeline.py. Use SOURCE_MODE='github' or 'kaggle_dataset'.")
    if str(selected).startswith("/kaggle/input"):
        selected = copy_source_to_working(selected)
    SRC_DIR = selected

PROJECT_ROOT = SRC_DIR.parent if SRC_DIR.name == "src" else SRC_DIR
os.chdir(SRC_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("Files in SRC_DIR:")
for path in sorted(SRC_DIR.iterdir()):
    print(" -", path.name)


## 2. Environment and Accelerator Check

Nếu CUDA không available, bật GPU trong Kaggle Notebook Settings. Nếu CUDA kernel lỗi, `train_pipeline.py` sẽ dừng ngay khi `DEVICE="cuda"` để tránh train nhầm trên CPU.


In [ ]:
import platform
import subprocess
import sys

if REINSTALL_TORCH_FOR_T4:
    cmd = [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
        *PYTORCH_PACKAGES,
        "--index-url", PYTORCH_CUDA_INDEX_URL,
    ]
    print("Installing Kaggle T4-compatible PyTorch wheel:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("PyTorch reinstall finished. If this cell replaced an already-imported torch, restart the notebook kernel and run all cells again.")

import torch
from torch import nn

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i), "capability=", torch.cuda.get_device_capability(i))
    smoke_device = torch.device("cuda:0")
    torch.cuda.set_device(smoke_device)
    probe = nn.Sequential(
        nn.Conv2d(3, 4, kernel_size=3, padding=1),
        nn.BatchNorm2d(4),
        nn.ReLU(inplace=True),
    ).to(smoke_device)
    x = torch.randn(2, 3, 16, 16, device=smoke_device)
    y = probe(x).mean()
    y.backward()
    torch.cuda.synchronize(smoke_device)
    print("CUDA smoke test: OK")
else:
    print("WARNING: GPU is not active; training will use CPU/fallback.")


## 3. Install Dependencies Without Replacing Kaggle PyTorch

Không cài đè `torch`, `torchvision`, `torchaudio` vì Kaggle đã có bản tương thích CUDA. Nếu đã từng cài đè torch trong session cũ, hãy Restart Session trước.


In [ ]:
import subprocess
import sys
from pathlib import Path

if INSTALL_DEPS:
    req = SRC_DIR / "requirements.txt"
    skip_packages = {"torch", "torchvision", "torchaudio"}
    filtered_req = Path("/kaggle/working/requirements_kaggle_no_torch.txt")
    if req.exists():
        filtered_lines = []
        for raw_line in req.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            package_name = line.split("==")[0].split(">=")[0].split("<=")[0].split("[")[0].lower()
            if package_name in skip_packages:
                print(f"Skipping PyTorch package managed by the GPU setup cell: {line}")
                continue
            filtered_lines.append(raw_line)
        filtered_req.write_text("\n".join(filtered_lines) + "\n", encoding="utf-8")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered_req)], check=True)
    else:
        packages = [
            "timm", "transformers", "monai", "pydicom", "nibabel", "SimpleITK",
            "scikit-learn", "faiss-cpu", "pandas", "matplotlib", "rich", "tqdm",
            "nltk", "rouge-score", "evaluate", "bert-score", "grad-cam", "qdrant-client",
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
else:
    print("Dependency installation skipped by config.")

import torch
print("Torch after dependency install:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0), "capability=", torch.cuda.get_device_capability(0))


## 4. Verify Imports and Compile Source


In [ ]:
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(SRC_DIR))

import torch
import pandas as pd
import numpy as np
from medical_vlm_pipeline import PipelineConfig, MedicalVLMPipeline, MedicalCaseDataset, load_iu_chest_xray_cases

print("Imports OK")
subprocess.run([sys.executable, "-m", "compileall", "medical_vlm_pipeline", "train_pipeline.py", "demo_pipeline.py"], check=True)

## 5. Dataset Sanity Check


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path(DATASET_DIR) / "cleaned_dataset.csv"
images_dir = Path(DATASET_DIR) / IMAGE_FOLDER
print("CSV path:", csv_path)
print("Images dir:", images_dir)
print("CSV exists:", csv_path.exists())
print("Images dir exists:", images_dir.exists())

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    display(df.head(3))
    cases = load_iu_chest_xray_cases(csv_path, images_dir)
    print("Loaded cases:", len(cases))
    if cases:
        print("First case:", cases[0])
else:
    print("WARNING: Dataset not found. Training will use synthetic fallback data.")

## 6. Optional Demo Pipeline


In [ ]:
import subprocess
import sys

if RUN_DEMO:
    subprocess.run([sys.executable, "demo_pipeline.py"], check=True)
else:
    print("Demo skipped by config.")

## 7. Full Training and Evaluation Test

Training stdout/stderr được lưu vào `report_kaggle/kaggle_train_stdout.log`. Nếu fail, cell sẽ in tail của log để thấy traceback thật.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if RUN_FULL_TRAIN:

    print("Training config:")
    print(f"  EPOCHS={EPOCHS}")
    print(f"  BATCH_SIZE={BATCH_SIZE}")
    print(f"  MULTI_GPU={MULTI_GPU}")
    print(f"  GPU_IDS={GPU_IDS}")
    print(f"  DEVICE={DEVICE}")
    print(f"  TARGET_COLUMN={TARGET_COLUMN}")
    print(f"  DERIVE_LABELS_FROM_REPORT={DERIVE_LABELS_FROM_REPORT}")
    print(f"  VAL_SPLIT={VAL_SPLIT}")
    print(f"  SPLIT_GROUP={SPLIT_GROUP}")
    print(f"  CLASS_WEIGHTING={CLASS_WEIGHTING}")
    if EPOCHS < 10:
        raise ValueError("EPOCHS is below 10. Re-run the top config cell after pull, or set EPOCHS = 150 before training.")
    cmd = [
        sys.executable, "train_pipeline.py",
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--device", DEVICE,
        "--dataset-dir", DATASET_DIR,
        "--synthetic-cases", str(SYNTHETIC_CASES),
        "--target-column", TARGET_COLUMN,
        "--val-split", str(VAL_SPLIT),
        "--split-group", SPLIT_GROUP,
        "--min-class-count", str(MIN_CLASS_COUNT),
        "--num-workers", str(NUM_WORKERS),
        "--multi-gpu", MULTI_GPU,
        "--gpu-ids", GPU_IDS,
        "--class-weighting", CLASS_WEIGHTING,
    ]
    if DERIVE_LABELS_FROM_REPORT:
        cmd.append("--derive-labels-from-report")
    if NO_AMP:
        cmd.append("--no-amp")
    if SKIP_PLOT:
        cmd.append("--skip-plot")

    log_dir = SRC_DIR / "report_kaggle"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / "kaggle_train_stdout.log"

    env = os.environ.copy()
    env.setdefault("CUDA_LAUNCH_BLOCKING", "1")
    env.setdefault("TOKENIZERS_PARALLELISM", "false")

    print("Running:", " ".join(cmd))
    print("Training log:", log_path)

    with open(log_path, "w", encoding="utf-8") as log_file:
        proc = subprocess.run(cmd, stdout=log_file, stderr=subprocess.STDOUT, text=True, env=env)

    log_text = log_path.read_text(encoding="utf-8", errors="replace")
    print("\n===== TRAIN LOG TAIL =====")
    print("\n".join(log_text.splitlines()[-240:]))
    print("===== END TRAIN LOG TAIL =====\n")

    if proc.returncode != 0:
        raise RuntimeError(
            f"Training failed with exit code {proc.returncode}. See full log at {log_path}."
        )
else:
    print("Full training skipped by config.")


## 8. Artifact Inspection

Các file này đủ để plot lại và reuse model/retriever mà không cần train lại.


In [ ]:
from pathlib import Path
import json
import pandas as pd

root = SRC_DIR
report_dir = root / "report_kaggle"
model_dir = report_dir / "model_artifacts"
artifact_paths = {
    "root_best_model": root / "best_model.pt",
    "best_model_state_dict": model_dir / "best_model_state_dict.pt",
    "final_model_state_dict": model_dir / "final_model_state_dict.pt",
    "best_training_checkpoint": model_dir / "best_training_checkpoint.pt",
    "final_training_checkpoint": model_dir / "final_training_checkpoint.pt",
    "model_metadata": model_dir / "model_metadata.json",
    "label_map": model_dir / "label_map.json",
    "retriever_index_state": model_dir / "retriever_index" / "faiss_retriever.state",
    "retriever_index_faiss": model_dir / "retriever_index" / "faiss_retriever.index",
    "run_config": report_dir / "run_config.json",
    "environment": report_dir / "environment.json",
    "case_index": report_dir / "case_index.csv",
    "train_case_index": report_dir / "train_case_index.csv",
    "validation_case_index": report_dir / "validation_case_index.csv",
    "training_metrics": report_dir / "training_metrics.csv",
    "epoch_metrics": report_dir / "epoch_metrics.json",
    "text_metrics": report_dir / "text_generation_metrics.json",
    "text_samples_csv": report_dir / "text_generation_samples.csv",
    "text_samples_json": report_dir / "text_generation_samples.json",
    "inference_report": report_dir / "inference_report.json",
    "run_summary": report_dir / "run_summary.json",
    "artifacts_manifest": report_dir / "artifacts_manifest.json",
    "training_curves": report_dir / "training_curves.png",
    "train_log": report_dir / "kaggle_train_stdout.log",
}

for name, path in artifact_paths.items():
    print(f"{name}: {path} | exists={path.exists()} | size={path.stat().st_size if path.exists() else 0}")

metrics_path = artifact_paths["training_metrics"]
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    display(metrics_df.tail())

text_samples_path = artifact_paths["text_samples_csv"]
if text_samples_path.exists():
    text_samples_df = pd.read_csv(text_samples_path)
    cols = ["case_id", "label", "predicted_diagnosis", "confidence", "bleu_1", "rouge_l", "num_retrieved"]
    display(text_samples_df[[c for c in cols if c in text_samples_df.columns]].head())

for key in ["model_metadata", "text_metrics", "inference_report", "run_summary"]:
    path = artifact_paths[key]
    if path.exists():
        print(f"\n--- {key} ---")
        with open(path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2)[:5000])


## 9. Display Training Curves


In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_path = SRC_DIR / "report_kaggle" / "training_curves.png"
if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print("No training_curves.png found. Check SKIP_PLOT or Matplotlib errors.")

## 10. Package Artifacts For Reuse

Cell này đóng gói model + logs + retriever index thành một file zip trong `/kaggle/working`. Bạn có thể tải file này về hoặc tạo Kaggle Dataset từ output để lần sau load lại model/retriever.


In [ ]:
import shutil
from pathlib import Path

package_base = Path("/kaggle/working/medical_vlm_pipeline_artifacts")
package_zip = Path(str(package_base) + ".zip")
report_dir = SRC_DIR / "report_kaggle"

if package_zip.exists():
    package_zip.unlink()

if report_dir.exists():
    shutil.make_archive(str(package_base), "zip", root_dir=report_dir)
    print("Created artifact package:", package_zip)
    print("Size MB:", round(package_zip.stat().st_size / (1024 * 1024), 2))
else:
    print("report_kaggle directory not found; run training first.")

## 11. Reload Notes

Artifacts quan trọng để dùng lại:

- `report_kaggle/model_artifacts/best_model_state_dict.pt`: weights tốt nhất.
- `report_kaggle/model_artifacts/final_model_state_dict.pt`: weights cuối run.
- `report_kaggle/model_artifacts/final_training_checkpoint.pt`: resume training gồm model/optimizer/scheduler/config.
- `report_kaggle/model_artifacts/retriever_index/faiss_retriever.state` và `.index`: vector DB/retriever đã build.
- `report_kaggle/model_artifacts/label_map.json`: mapping label.
- `report_kaggle/training_metrics.csv` và `epoch_metrics.json`: plot lại kết quả.
- `medical_vlm_pipeline_artifacts.zip`: gói output để tải/reuse.
